In [6]:
import pandas as pd
import os


In [7]:
def compute_12m_return(df):
    """Calculate 12-month return based on 'Adj Close'"""
    return df['Adj Close'].pct_change(252)

def generate_12m_signal(df):
    """
    Create signal:
    +1 if 12m return > 0
    -1 if 12m return < 0
     0 if 12m return == 0 or NaN
    """
    df['signal'] = 0
    df.loc[df['12m_return'] > 0, 'signal'] = 1
    df.loc[df['12m_return'] < 0, 'signal'] = -1
    return df


In [11]:
import pandas as pd
import os

raw_dir = "data/raw"
data = {}

for file in os.listdir(raw_dir):
    if file.endswith(".parquet"):
        ticker = file.replace(".parquet", "")
        filepath = os.path.join(raw_dir, file)
        try:
            df = pd.read_parquet(filepath)
            data[ticker] = df
            print(f"Loaded {ticker} with {len(df)} rows.")
        except Exception as e:
            print(f"Failed to load {ticker}: {e}")

print(f"Tickers loaded: {list(data.keys())}")

Loaded AAPL with 751 rows.
Loaded AGG with 751 rows.
Loaded BTC-USD with 1095 rows.
Loaded EURUSD with 780 rows.
Loaded EURUSD=X with 781 rows.
Loaded EURUSDX with 779 rows.
Loaded GLD with 751 rows.
Loaded IWM with 751 rows.
Loaded MSFT with 751 rows.
Loaded QQQ with 751 rows.
Loaded SLV with 751 rows.
Loaded SPY with 751 rows.
Loaded spy_1d_raw with 32 rows.
Loaded TLT with 751 rows.
Loaded USO with 751 rows.
Loaded VIX with 751 rows.
Loaded XLF with 751 rows.
Loaded XLV with 751 rows.
Loaded ^VIX with 752 rows.
Tickers loaded: ['AAPL', 'AGG', 'BTC-USD', 'EURUSD', 'EURUSD=X', 'EURUSDX', 'GLD', 'IWM', 'MSFT', 'QQQ', 'SLV', 'SPY', 'spy_1d_raw', 'TLT', 'USO', 'VIX', 'XLF', 'XLV', '^VIX']


In [18]:
import os

processed_dir = "data/processed"
os.makedirs(processed_dir, exist_ok=True)

for ticker, df in data.items():
    try:
        # Try MultiIndex column first
        price_col = ('Close', ticker)
        
        if price_col in df.columns:
            price_series = df[price_col]
        elif 'Close' in df.columns:
            # fallback if columns are single level
            price_series = df['Close']
        elif 'close' in df.columns:
            price_series = df['close']
        else:
            raise ValueError("No suitable Close price column found")
        
        df['12m_return'] = price_series.pct_change(252, fill_method=None)
        df['signal'] = 0
        df.loc[df['12m_return'] > 0, 'signal'] = 1
        df.loc[df['12m_return'] < 0, 'signal'] = -1
        
        save_path = os.path.join(processed_dir, f"{ticker}.parquet")
        df.to_parquet(save_path)
        print(f"Saved processed {ticker} to {save_path}")
    except Exception as e:
        print(f"Error processing {ticker}: {e}")


Saved processed AAPL to data/processed\AAPL.parquet
Saved processed AGG to data/processed\AGG.parquet
Saved processed BTC-USD to data/processed\BTC-USD.parquet
Saved processed EURUSD to data/processed\EURUSD.parquet
Saved processed EURUSD=X to data/processed\EURUSD=X.parquet
Saved processed EURUSDX to data/processed\EURUSDX.parquet
Saved processed GLD to data/processed\GLD.parquet
Saved processed IWM to data/processed\IWM.parquet
Saved processed MSFT to data/processed\MSFT.parquet
Saved processed QQQ to data/processed\QQQ.parquet
Saved processed SLV to data/processed\SLV.parquet
Saved processed SPY to data/processed\SPY.parquet
Saved processed spy_1d_raw to data/processed\spy_1d_raw.parquet
Saved processed TLT to data/processed\TLT.parquet
Saved processed USO to data/processed\USO.parquet
Saved processed VIX to data/processed\VIX.parquet
Saved processed XLF to data/processed\XLF.parquet
Saved processed XLV to data/processed\XLV.parquet
Saved processed ^VIX to data/processed\^VIX.parque

In [19]:
print("Processed files:")
print(os.listdir(processed_dir))


Processed files:
['AAPL.parquet', 'AGG.parquet', 'BTC-USD.parquet', 'EURUSD.parquet', 'EURUSD=X.parquet', 'EURUSDX.parquet', 'GLD.parquet', 'IWM.parquet', 'MSFT.parquet', 'QQQ.parquet', 'SLV.parquet', 'SPY.parquet', 'spy_1d_raw.parquet', 'TLT.parquet', 'USO.parquet', 'VIX.parquet', 'XLF.parquet', 'XLV.parquet', '^VIX.parquet']
